# Wyoming Statewide Civic Cartography & Forensic Audit — Colab Post-Processing

**Architecture Role (Phase 2 — Post-Processing & Analytics):**
- **Phase 1 (GitHub Actions Matrix):** `.github/workflows/passive_cartography.yml` runs parallel passive DNS, Certificate Transparency (`crt.sh`), and polite HTTPS SaaS fingerprinting across `wyo-gov` + all 23 Wyoming counties (`fail-fast: false`), producing `wyoming-statewide-records-infrastructure`.
- **Phase 2 (This Colab Notebook):** Loads the consolidated SQLite bundle (or builds an offline baseline directly from `wyoming/targets.json`), queries vendor classifications with `sqlite3`/`pandas`, runs the Seven-Section Forensic Audit runners (`promethean.forensic_audit`), and exports human-reviewed W.S. 16-4-202 draft packets.

> **Guardrails:** Passive OSINT & public records only (`LEGAL.md`). All anomaly flags are `candidate_unreviewed` audit leads requiring certified-record verification; all W.S. 16-4-202 outputs are unsent drafts (`sent_by_this_tool = False`).

In [ ]:
import json
import os
from pathlib import Path
import sqlite3
import subprocess
import sys

# Clone the repository if running inside a fresh Google Colab VM
if not Path('wyoming/targets.json').exists():
    if not Path('therealwindycity').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bartimoussmith-oss/therealwindycity.git'], check=True)
    os.chdir('therealwindycity')

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from promethean.forensic_audit import (
    MIRROR_LEDGERS,
    OPERATIONAL_SECTIONS,
    audit_check_register_csv,
    audit_ws_16_4_202_timeline,
    package_osint_artifact,
    verify_osint_provenance,
)
from tools.passive_cartography import consolidate_results, load_targets, scan_jurisdiction

print('Loaded targets:', len(load_targets()['targets']), '| Mirror ledgers:', len(MIRROR_LEDGERS))

In [ ]:
# Load a downloaded GitHub Actions bundle DB if uploaded to Colab, or build an offline baseline
DB_FILE = Path('cartography-results/wyoming_statewide_records_infrastructure.db')
if not DB_FILE.exists():
    raw_dir = Path('cartography-results/raw')
    for t in load_targets()['targets']:
        scan_jurisdiction(t, output_dir=raw_dir, offline=True)
    consolidate_results(
        input_dir=raw_dir,
        db_out=DB_FILE,
        summary_json_out=Path('cartography-results/consolidated_cartography.json'),
        markdown_out=Path('cartography-results/vendor_pra_matrix.md'),
    )

conn = sqlite3.connect(DB_FILE)
display(pd.read_sql_query(
    'SELECT jurisdiction, name, type, county_seat_name, verification, collection_mode, vendor_count, subdomain_count '
    'FROM jurisdiction_targets ORDER BY CASE WHEN type="state-portal" THEN 0 ELSE 1 END, jurisdiction',
    conn,
))

In [ ]:
# Inspect the 8 Cross-Jurisdictional State & Federal Mirror Ledgers
display(pd.read_sql_query('SELECT layer_id, slug, name, endpoints_json, sections_json, exposes_summary FROM mirror_ledger_catalog', conn))

# Inspect Vendor-Specific W.S. 16-4-202 Draft Templates (unsent, human-reviewed)
display(pd.read_sql_query(
    'SELECT jurisdiction, template_class, section_id, substr(draft_text, 1, 180) AS draft_preview '
    'FROM pra_draft_templates LIMIT 15',
    conn,
))

In [ ]:
# Section 2 & Section 7 Forensic Runner Smoke Example (replace CSV path with an exported check register)
pra_check = audit_ws_16_4_202_timeline(
    submitted_iso='2026-09-01',
    ack_iso='2026-09-15',
    fulfilled_iso='2026-10-10',
)
print(json.dumps(pra_check, indent=2))